In [2]:
from google.colab import files
uploaded = files.upload()

Saving car1.zip to car1.zip
Saving car2.zip to car2.zip
Saving car3.zip to car3.zip


In [3]:
import zipfile
import os

os.makedirs('dataset/car1', exist_ok=True)
os.makedirs('dataset/car2', exist_ok=True)
os.makedirs('dataset/car3', exist_ok=True)

with zipfile.ZipFile('car1.zip', 'r') as z:
    z.extractall('dataset/car1')

with zipfile.ZipFile('car2.zip', 'r') as z:
    z.extractall('dataset/car2')

with zipfile.ZipFile('car3.zip', 'r') as z:
    z.extractall('dataset/car3')

for car in ['car1', 'car2', 'car3']:
    print(f"{car}: {len(os.listdir(f'dataset/{car}'))} images")

car1: 21 images
car2: 22 images
car3: 20 images


In [4]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from PIL import Image
from sklearn.model_selection import train_test_split

dataset_path = 'dataset'
class_names = ['car1', 'car2', 'car3']



In [5]:
X = [] #images
y = [] #labels

for label, class_name in enumerate(class_names):
    class_folder = os.path.join(dataset_path, class_name)
    for img_name in os.listdir(class_folder):
        img_path = os.path.join(class_folder, img_name)
        img = Image.open(img_path).convert('RGB')
        img = img.resize((32, 32))
        img_array = np.array(img)
        X.append(img_array)
        y.append(label)

X = np.array(X)
y = np.array(y)

In [6]:
X = X/255.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
cnn = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')

])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
cnn.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
history = cnn.fit(X_train, y_train, epochs=50, batch_size = 8, validation_data=(X_test, y_test))

Epoch 1/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.3000 - loss: 1.1175 - val_accuracy: 0.2308 - val_loss: 1.1050
Epoch 2/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3400 - loss: 1.0635 - val_accuracy: 0.2308 - val_loss: 1.1099
Epoch 3/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4600 - loss: 1.0405 - val_accuracy: 0.3846 - val_loss: 1.0270
Epoch 4/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5000 - loss: 0.9717 - val_accuracy: 0.5385 - val_loss: 1.0455
Epoch 5/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.6200 - loss: 0.9049 - val_accuracy: 0.4615 - val_loss: 1.0838
Epoch 6/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6400 - loss: 0.7966 - val_accuracy: 0.3846 - val_loss: 0.9620
Epoch 7/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7200 - loss: 0.6977 - val_accuracy: 0.4615 - val_loss: 1.0506
Epoch 8/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8000 - loss: 0.5675 - val_accuracy: 0.4615 - val_loss: 0.9566


In [10]:
from google.colab import files
from PIL import Image
import numpy as np

class_names = ['Opel Astra', 'Ford Escort', 'Ford F350']

print("Upload an image to identify:")
pred_upload = files.upload()

pred_filename = list(pred_upload.keys())[0]
pred_img = Image.open(pred_filename).convert('RGB')
pred_img = pred_img.resize((32, 32))
pred_array = np.array(pred_img) / 255.0
pred_array = pred_array.reshape(1, 32, 32, 3)

prediction = cnn.predict(pred_array)
predicted_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction) * 100

print(f"\nCar: {predicted_class} (Confidence: {confidence:.2f}%)")

Upload an image to identify:


Saving 20260604_175849.jpg to 20260604_175849.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

Car: Ford Escort (Confidence: 99.99%)
